# Ders 2B — Feature Selection

En iyi feature set üzerinde top-k feature selection denenir.

## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.

required_packages = [
    ("pandas", "pandas"),  # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),  # Sayısal matris ve vektör işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),  # Grafik çizimleri için kullanılır.
    ("sklearn", "scikit-learn"),  # Makine öğrenmesi modelleri ve metrikleri için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri, indeksleme ve skor sıralama için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Feature selection performans grafiklerini çizmek için kullanılır.

from sklearn.model_selection import train_test_split
# Train/test ayrımı yapmak için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# Feature selection sonuçlarını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import RandomForestClassifier
# Gatekeeper model ve model-based feature importance için RandomForest kullanılır.
from sklearn.preprocessing import MinMaxScaler
# Chi-square yöntemi negatif olmayan feature istediği için gerektiğinde ölçekleme yapar.
from sklearn.feature_selection import f_classif, chi2, mutual_info_classif
# ANOVA, Chi-square ve Mutual Information feature selection yöntemleri çağırılır.


DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı ve modeller için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır ve sadece bu scriptte ihtiyaç duyulan işlemleri kapsar.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da tablo olarak, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display daha okunabilir tablo üretir.
        display(df.head(n))
    except NameError:
        # Terminalde display yoksa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def load_feature_table(dataset_key):
    """Seçilen veri setinin hazır feature CSV dosyasını GitHub raw linkinden okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait URL ve kısa isim bilgisi alınır.
    df = pd.read_csv(info["feature_url"])
    # Feature CSV doğrudan GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan tablonun boyutu ekrana yazdırılır.
    return df
    # Okunan tablo modelleme adımlarına gönderilir.


def feature_columns(df, feature_set):
    """İstenen feature ailesine ait kolonları seçer."""
    prefix_map = {
        # Feature set adları CSV kolon prefixleriyle eşleştirilir.
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    prefixes = prefix_map[feature_set]
    # İstenen feature setinin kolon prefixleri alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Prefix ile eşleşen feature kolonları seçilir.
    if not cols:
        # Hiç kolon bulunmazsa erken ve anlaşılır hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.


def available_feature_sets(df):
    """Feature CSV içinde gerçekten bulunan feature setlerini listeler."""
    sets = ["morgan", "maccs", "rdkit", "maccs_morgan", "maccs_rdkit", "morgan_rdkit", "all_available"]
    # Temel feature setleri ve birleşik feature setleri sırayla denenir.
    if any(c.startswith("Avalon_") for c in df.columns):
        # Avalon kolonları varsa Avalon tek başına da aday feature set olarak eklenir.
        sets.insert(3, "avalon")
    return sets
    # Bu veri setinde denenebilecek feature set listesi döndürülür.


def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .fillna(0.0)
        # NaN değerler model eğitimi bozulmasın diye 0 yapılır.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_data(df, cols):
    """Sınıf oranını koruyarak train/test ayrımı yapar."""
    X, y = make_xy(df, cols)
    # Seçilen featurelardan X ve y hazırlanır.
    return train_test_split(X, y, df, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    # Stratify ile sınıf oranı train/test içinde korunur.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        return model.decision_function(X)
    return model.predict(X).astype(float)
    # Son seçenek olarak sınıf tahmini skor gibi kullanılır.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def plot_metric_bar(df, label_col, metric, title, out_file=None, top_n=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # En iyi sonuçlar üstte olacak şekilde metrik değerine göre sıralanır.
    if top_n is not None:
        # Çok uzun tablolar için ilk n sonuç çizilebilir.
        plot_df = plot_df.head(top_n)
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her model/feature set için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    if metric == "ROC":
        # ROC-AUC değerleri birbirine yakın olduğunda farklar daha net görünsün diye grafik 0.60'tan başlatılır.
        plt.xlim(left=0.60)
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


def add_gain(current_df, previous_df, current_name, previous_name):
    """Bir önceki adıma göre ROC/AP/F1 farkını ve yüzde değişimi hesaplar."""
    rows = []
    # Hesaplanan karşılaştırma satırları burada biriktirilir.
    for _, current in current_df.iterrows():
        # Mevcut adımın her veri seti sonucu tek tek dolaşılır.
        dataset = current["Dataset"]
        # Karşılaştırılacak veri seti adı alınır.
        prev = previous_df[previous_df["Dataset"] == dataset].sort_values("ROC", ascending=False).iloc[0]
        # Aynı veri setinin önceki adımdaki en iyi sonucu alınır.
        row = current.to_dict()
        # Mevcut sonuç satırı sözlüğe çevrilir.
        row["CurrentStep"] = current_name
        # Mevcut adım adı eklenir.
        row["ComparedWith"] = previous_name
        # Karşılaştırılan önceki adım adı eklenir.
        row["Previous_ROC"] = prev["ROC"]
        # Önceki ROC değeri tabloya eklenir.
        row["ROC_Delta"] = current["ROC"] - prev["ROC"]
        # Mutlak ROC farkı hesaplanır.
        row["ROC_Gain_%"] = 100 * (current["ROC"] - prev["ROC"]) / abs(prev["ROC"]) if abs(prev["ROC"]) > 1e-12 else np.nan
        # ROC yüzde değişimi hesaplanır.
        row["Previous_AP"] = prev["AP"]
        # Önceki AP değeri tabloya eklenir.
        row["AP_Delta"] = current["AP"] - prev["AP"]
        # AP farkı hesaplanır.
        row["AP_Gain_%"] = 100 * (current["AP"] - prev["AP"]) / abs(prev["AP"]) if abs(prev["AP"]) > 1e-12 else np.nan
        # AP yüzde değişimi hesaplanır.
        row["Previous_F1"] = prev["F1"]
        # Önceki F1 değeri tabloya eklenir.
        row["F1_Delta"] = current["F1"] - prev["F1"]
        # F1 farkı hesaplanır.
        row["F1_Gain_%"] = 100 * (current["F1"] - prev["F1"]) / abs(prev["F1"]) if abs(prev["F1"]) > 1e-12 else np.nan
        # F1 yüzde değişimi hesaplanır.
        rows.append(row)
        # Karşılaştırma satırı listeye eklenir.
    return pd.DataFrame(rows)
    # Tüm karşılaştırmalar tablo olarak döndürülür.


def make_random_forest():
    """Baseline ve gatekeeper için RandomForest modeli kurar."""
    return RandomForestClassifier(
        n_estimators=300,
        # 300 ağaç, eğitim süresi ve stabilite arasında dengeli bir seçimdir.
        max_features="sqrt",
        # Her bölünmede featureların karekökü kadar aday denenir.
        class_weight="balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        random_state=RANDOM_STATE,
        # Tekrarlanabilirlik sağlar.
        n_jobs=-1,
        # Mevcut tüm işlemciler kullanılır.
    )


def feature_ranking(method, X_train, y_train, feature_names):
    """Feature selection yöntemi için feature sıralaması üretir."""
    if method == "none":
        # Seçim yoksa mevcut feature sırası aynen kullanılır.
        return list(range(len(feature_names)))

    if method == "ANOVA":
        # ANOVA F-score her feature ile target arasındaki tekil ilişkiyi ölçer.
        scores, _ = f_classif(X_train, y_train)
        # Her feature için F skoru hesaplanır.
        return np.argsort(np.nan_to_num(scores))[::-1].tolist()
        # Featurelar yüksek skordan düşük skora sıralanır.

    if method == "Chi2":
        # Chi-square negatif olmayan feature değerleri ister.
        X_for_chi2 = X_train
        # Başlangıçta orijinal X kullanılır.
        if np.nanmin(X_for_chi2) < 0:
            # Negatif değer varsa 0-1 aralığına ölçeklenir.
            X_for_chi2 = MinMaxScaler().fit_transform(X_for_chi2)
        scores, _ = chi2(X_for_chi2, y_train)
        # Chi-square skorları hesaplanır.
        return np.argsort(np.nan_to_num(scores))[::-1].tolist()
        # Featurelar yüksek skordan düşük skora sıralanır.

    if method == "MutualInfo":
        # Mutual information doğrusal olmayan ilişkileri de yakalayabilir.
        scores = mutual_info_classif(X_train, y_train, discrete_features="auto", random_state=RANDOM_STATE)
        # Her feature için mutual information skoru hesaplanır.
        return np.argsort(np.nan_to_num(scores))[::-1].tolist()
        # Featurelar yüksek skordan düşük skora sıralanır.

    if method == "RF_importance":
        # Model-based seçim için RandomForest feature importance kullanılır.
        selector_model = make_random_forest()
        # Selector olarak RF modeli kurulur.
        selector_model.fit(X_train, y_train)
        # RF modeli feature önemlerini öğrenmek için eğitilir.
        return np.argsort(selector_model.feature_importances_)[::-1].tolist()
        # Featurelar importance değerine göre sıralanır.

    raise ValueError(f"Bilinmeyen feature selection yöntemi: {method}")

## 4. Feature ablation ve feature selection

Bu notebook kendi içinde önce en iyi feature seti bulur, sonra o feature set üzerinde top-k feature selection yapar.

In [ ]:
selection_methods = ["ANOVA", "Chi2", "MutualInfo", "RF_importance"]
# Denenecek dört feature selection yöntemi tanımlanır.
dataset_k_values = {
    "ERa_BLA_assay": [50, 100, 150, 200],
    "ERa_LUC_VM7_assay": [250, 750, 1250, 1750],
}
# Her veri seti için uygun top-k feature sayıları ayrı tanımlanır.

forced_feature_sets = {
    "ERa_BLA_assay": "rdkit",
    "ERa_LUC_VM7_assay": "all_available",
}
# Step 3 sonucuna göre feature selection yapılacak feature setler sabitlenir.
# ERa_BLA_assay için yalnızca RDKit descriptor featureları kullanılır.
# ERa_LUC_VM7_assay için tüm mevcut featurelar kullanılır.

best_selection_rows = []
# Her veri seti için en iyi feature selection sonucu burada tutulur.
best_gatekeeper_rows = []
# Step 3'ten gelen en iyi gatekeeper feature set sonuçları burada tutulur.
comparison_table_rows = []
# Her veri seti için gatekeeper vs feature selection karşılaştırma satırları burada tutulur.
all_selection_rows = []
# Tüm feature selection yöntemleri ve tüm k değerleri burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    feature_ablation_rows = []
    # Bu veri seti için feature ablation sonuçları burada tutulur.
    for feature_set in available_feature_sets(df):
        # Mevcut feature setleri sırayla denenir.
        feature_cols = feature_columns(df, feature_set)
        # İlgili feature setin kolonları seçilir.
        X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, feature_cols)
        # Seçilen feature setiyle train/test ayrımı yapılır.

        model = make_random_forest()
        # Feature set karşılaştırması için RandomForest modeli kurulur.
        model.fit(X_train, y_train)
        # Model train set üzerinde eğitilir.

        y_pred = model.predict(X_test)
        # Test seti için sınıf tahmini alınır.
        y_score = class1_score(model, X_test)
        # ROC/AP için class 1 skoru alınır.

        metrics = metric_dict(y_test, y_pred, y_score)
        # Test performans metrikleri hesaplanır.
        metrics.update({
            "Dataset": dataset_key,
            "FeatureSet": feature_set,
            "FeatureCount": len(feature_cols),
            "ModelType": "RandomForest",
        })
        # Sonuç satırına feature set bilgileri eklenir.
        feature_ablation_rows.append(metrics)
        # Sonuç listeye eklenir.

    feature_ablation_df = pd.DataFrame(feature_ablation_rows).sort_values("ROC", ascending=False)
    # Feature ablation sonuçları ROC-AUC değerine göre sıralanır.
    best_feature_set = forced_feature_sets[dataset_key]
    # Bu veri seti için Step 3'ten taşınacak feature set sabit listeden alınır.
    best_feature = feature_ablation_df[feature_ablation_df["FeatureSet"] == best_feature_set].iloc[0]
    # Sabitlenen feature setin Step 3 gatekeeper performansı alınır.
    best_gatekeeper_rows.append(best_feature.to_dict())
    # Step 3 gatekeeper sonucu daha sonra karşılaştırma için saklanır.

    base_features = feature_columns(df, best_feature_set)
    # Feature selection sadece Step 3'te seçilen en iyi feature set üzerinde yapılır.
    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, base_features)
    # Feature selection için train/test ayrımı yapılır.

    k_values = dataset_k_values[dataset_key]
    # Bu veri seti için kullanılacak top-k değerleri alınır.

    selection_rows = []
    # Bu veri setindeki bütün selection kombinasyonları burada tutulur.
    for method in selection_methods:
        # Dört feature selection yöntemi sırayla denenir.
        ranking = feature_ranking(method, X_train, y_train, base_features)
        # Featurelar seçilen yönteme göre sıralanır.

        for k in k_values:
            # Her top-k değeri sırayla denenir.
            k_effective = min(k, len(base_features))
            # İstenen k değeri feature sayısını aşarsa mevcut maksimum feature sayısına çekilir.
            idx = ranking[:k_effective]
            # Sıralamadaki ilk k feature indexi alınır.
            selected_features = [base_features[i] for i in idx]
            # Seçilen feature isimleri oluşturulur.

            model = make_random_forest()
            # Selection denemesinde gatekeeper RF modeli kullanılır.
            model.fit(X_train[:, idx], y_train)
            # Model sadece seçilen featurelarla eğitilir.

            y_pred = model.predict(X_test[:, idx])
            # Test seti için sınıf tahmini alınır.
            y_score = class1_score(model, X_test[:, idx])
            # ROC/AP için class 1 skoru alınır.

            metrics = metric_dict(y_test, y_pred, y_score)
            # Test metrikleri hesaplanır.
            metrics.update({
                "Dataset": dataset_key,
                "FeatureSet": best_feature_set,
                "SelectionMethod": method,
                "K": k_effective,
                "RequestedK": k,
                "FeatureCount": len(selected_features),
                "SelectedFeatures": selected_features,
                "Previous_ROC_from_FeatureAblation": best_feature["ROC"],
                "SelectionLabel": method + "_top" + str(k_effective),
            })
            # Sonuç satırına selection yöntemi, k değeri, grafik etiketi ve feature bilgileri eklenir.
            selection_rows.append(metrics)
            # Selection sonucu veri seti listesine eklenir.
            all_selection_rows.append(metrics)
            # Selection sonucu genel listeye eklenir.

    selection_df = pd.DataFrame(selection_rows).sort_values("ROC", ascending=False)
    # Feature selection denemeleri ROC-AUC değerine göre sıralanır.
    selection_df["SelectionLabel"] = selection_df["SelectionMethod"] + "_top" + selection_df["K"].astype(str)
    # Grafik ve tablo için yöntem + k etiketi oluşturulur.
    best_selection = selection_df.iloc[0].to_dict()
    # En iyi feature selection sonucu ROC-AUC değerine göre seçilir.
    best_selection_rows.append(best_selection)
    # Veri seti için en iyi selection sonucu saklanır.

    reference_row = {
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper feature set",
        "ModelType": "RandomForest",
        "FeatureSet": best_feature["FeatureSet"],
        "SelectionMethod": "none",
        "K": int(best_feature["FeatureCount"]),
        "FeatureCount": int(best_feature["FeatureCount"]),
        "ROC": best_feature["ROC"],
        "AP": best_feature["AP"],
        "F1": best_feature["F1"],
        "ROC_Delta_vs_Gatekeeper": 0.0,
        "ROC_Gain_%": 0.0,
        "AP_Delta_vs_Gatekeeper": 0.0,
        "AP_Gain_%": 0.0,
        "F1_Delta_vs_Gatekeeper": 0.0,
        "F1_Gain_%": 0.0,
    }
    # Karşılaştırma tablosunun ilk satırı Step 3'te seçilen gatekeeper sonucudur.

    selection_row = {
        "Dataset": dataset_key,
        "Comparison": "Best feature selection",
        "ModelType": "RandomForest",
        "FeatureSet": best_selection["FeatureSet"],
        "SelectionMethod": best_selection["SelectionMethod"],
        "K": int(best_selection["K"]),
        "FeatureCount": int(best_selection["FeatureCount"]),
        "ROC": best_selection["ROC"],
        "AP": best_selection["AP"],
        "F1": best_selection["F1"],
        "ROC_Delta_vs_Gatekeeper": best_selection["ROC"] - best_feature["ROC"],
        "ROC_Gain_%": 100 * (best_selection["ROC"] - best_feature["ROC"]) / abs(best_feature["ROC"]) if abs(best_feature["ROC"]) > 1e-12 else np.nan,
        "AP_Delta_vs_Gatekeeper": best_selection["AP"] - best_feature["AP"],
        "AP_Gain_%": 100 * (best_selection["AP"] - best_feature["AP"]) / abs(best_feature["AP"]) if abs(best_feature["AP"]) > 1e-12 else np.nan,
        "F1_Delta_vs_Gatekeeper": best_selection["F1"] - best_feature["F1"],
        "F1_Gain_%": 100 * (best_selection["F1"] - best_feature["F1"]) / abs(best_feature["F1"]) if abs(best_feature["F1"]) > 1e-12 else np.nan,
    }
    # Karşılaştırma tablosunun ikinci satırı feature selection sonrası ROC-AUC temelli en iyi sonuçtur.

    comparison_table_rows.extend([reference_row, selection_row])
    # İki satırlık karşılaştırma genel listeye eklenir.

best_gatekeeper_df = pd.DataFrame(best_gatekeeper_rows)
# Step 3 gatekeeper sonuçları tabloya çevrilir.
best_selection_df = pd.DataFrame(best_selection_rows)
# En iyi selection sonuçları tabloya çevrilir.
all_selection_df = pd.DataFrame(all_selection_rows)
# Bütün method-k kombinasyonları tabloya çevrilir.
comparison_df = pd.DataFrame(comparison_table_rows)
# Gatekeeper ve feature selection karşılaştırma satırları tabloya çevrilir.

show_table(
    best_selection_df[["Dataset", "FeatureSet", "SelectionMethod", "K", "FeatureCount", "ROC", "AP", "F1"]],
    title="Bu notebook için yeniden belirlenen en iyi feature selection ayarı"
)
# En iyi selection ayarları ekranda gösterilir.

lesson_out = OUTPUT_ROOT / "04_feature_selection"
# Feature selection çıktıları için klasör belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

best_gatekeeper_df.to_csv(lesson_out / "04_step3_best_gatekeeper_per_dataset.csv", index=False)
# Step 3'ten gelen en iyi gatekeeper feature set sonuçları CSV olarak kaydedilir.
best_selection_df.to_csv(lesson_out / "04_best_feature_selection_per_dataset.csv", index=False)
# En iyi feature selection sonucu CSV olarak kaydedilir.
all_selection_df.to_csv(lesson_out / "04_all_feature_selection_combinations.csv", index=False)
# Bütün feature selection method-k kombinasyonları CSV olarak kaydedilir.
comparison_df.to_csv(lesson_out / "04_gatekeeper_vs_best_feature_selection_all.csv", index=False)
# Gatekeeper ve en iyi feature selection karşılaştırması CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # İki classification veri seti için ayrı kombinasyon tabloları ve ayrı karşılaştırma tabloları üretilir.
    dataset_selection_table = all_selection_df[all_selection_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin bütün feature selection kombinasyonları alınır.
    dataset_selection_table = dataset_selection_table.sort_values("ROC", ascending=False)
    # Kombinasyon tablosu ROC-AUC değerine göre sıralanır.
    if "SelectionLabel" not in dataset_selection_table.columns:
        # Eski çalıştırma/cache kaynaklı etiket kolonu yoksa burada yeniden oluşturulur.
        dataset_selection_table["SelectionLabel"] = dataset_selection_table["SelectionMethod"] + "_top" + dataset_selection_table["K"].astype(str)
    dataset_selection_table.to_csv(lesson_out / f"{dataset_key}_all_feature_selection_combinations.csv", index=False)
    # İlgili veri setinin bütün kombinasyon tablosu CSV olarak kaydedilir.

    show_table(
        dataset_selection_table[
            [
                "Dataset",
                "FeatureSet",
                "SelectionMethod",
                "K",
                "FeatureCount",
                "ROC",
                "AP",
                "F1",
                "Accuracy",
                "BalancedAccuracy",
                "Recall",
                "Specificity",
                "Precision",
            ]
        ],
        title=f"{dataset_key} — bütün feature selection kombinasyonları"
    )
    # İlgili veri seti için 4 method x 4 k kombinasyonunun tamamı ekranda gösterilir.

    plot_metric_bar(
        dataset_selection_table,
        label_col="SelectionLabel",
        metric="ROC",
        title=f"{dataset_key} — feature selection kombinasyonları ROC-AUC",
        out_file=lesson_out / f"{dataset_key}_all_feature_selection_combinations_roc.png",
    )
    # İlgili veri seti için bütün feature selection kombinasyonları ROC-AUC bar chart olarak çizilir.

    dataset_comparison_table = comparison_df[comparison_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin gatekeeper ve feature selection satırları alınır.
    dataset_comparison_table.to_csv(lesson_out / f"{dataset_key}_gatekeeper_vs_best_feature_selection.csv", index=False)
    # İlgili veri setinin iki satırlık karşılaştırma tablosu CSV olarak kaydedilir.

    show_table(
        dataset_comparison_table[
            [
                "Dataset",
                "Comparison",
                "ModelType",
                "FeatureSet",
                "SelectionMethod",
                "K",
                "FeatureCount",
                "ROC",
                "AP",
                "F1",
                "ROC_Delta_vs_Gatekeeper",
                "ROC_Gain_%",
                "AP_Delta_vs_Gatekeeper",
                "AP_Gain_%",
                "F1_Delta_vs_Gatekeeper",
                "F1_Gain_%",
            ]
        ],
        title=f"{dataset_key} — Gatekeeper feature set vs en iyi feature selection"
    )
    # Her veri seti için ayrı gatekeeper vs en iyi feature selection tablosu ekranda gösterilir.
